In [ ]:
from google.colab import drive   # connect colab to google drive for file transferring
drive.mount('/content/drive')

In [ ]:
!pip install fair-esm

In [ ]:
!pip install biopython

In [ ]:
import torch
import esm
import os
import numpy as np
import pandas as pd
from Bio import PDB

# Base path for the folds
base_fold_path = '/content/drive/MyDrive/@antigen_epitope/abdb_combine'
new_fold_path = '/content/drive/MyDrive/@antigen_epitope/abdb_combine'

In [ ]:
# Load the pre-trained ESM-2 model and its alphabet
esm2_model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()
esm2_model.eval()  # Disables dropout for deterministic results

In [ ]:
def get_esm2_embedding(sequence, model, alphabet):
    batch_converter = alphabet.get_batch_converter()
    data = [("protein", sequence)]
    batch_labels, batch_strs, batch_tokens = batch_converter(data)
    batch_lens = (batch_tokens != alphabet.padding_idx).sum(1)

    with torch.no_grad():
        results = model(batch_tokens, repr_layers=[6], return_contacts=True)
    token_representations = results["representations"][6]

    sequence_embedding = token_representations[0, 1 : batch_lens[0] - 1]
    return sequence_embedding

In [ ]:
# Map three-letter amino acid codes to one-letter codes
aa_map = {
    'ALA': 'A', 'ARG': 'R', 'ASN': 'N', 'ASP': 'D', 'CYS': 'C',
    'GLU': 'E', 'GLN': 'Q', 'GLY': 'G', 'HIS': 'H', 'ILE': 'I',
    'LEU': 'L', 'LYS': 'K', 'MET': 'M', 'PHE': 'F', 'PRO': 'P',
    'SER': 'S', 'THR': 'T', 'TRP': 'W', 'TYR': 'Y', 'VAL': 'V'
}

input_dir = os.path.join(base_fold_path, 'data', 'antigen_info_SI_interaction')
output_dir = os.path.join(new_fold_path, 'data', 'all_antigen_info_SI')
os.makedirs(output_dir, exist_ok=True)

# Process each TSV file in the input directory
for tsv_filename in os.listdir(input_dir):
    if tsv_filename.endswith('.tsv'):
        file_path = os.path.join(input_dir, tsv_filename)
        df = pd.read_csv(file_path, sep='\t')

        # Convert three-letter AA codes to one-letter sequence
        sequence = ''.join(df['AA'].map(aa_map))

        # Get the embeddings for the sequence
        embeddings = get_esm2_embedding(sequence, esm2_model, alphabet)

        embedding_df = pd.DataFrame(embeddings, columns=[f'embedding_{i}' for i in range(embeddings.shape[1])])
        df = pd.concat([df, embedding_df], axis=1)

        output_file = os.path.join(output_dir, f"{tsv_filename}_with_embeddings.tsv")
        df.to_csv(output_file, sep='\t', index=False)


In [ ]:
df

In [ ]:
df_x = pd.read_csv('/content/drive/MyDrive/@antigen_epitope/abdb_combine/data_X/all_data_with_SI.csv')
df_y = pd.read_csv('/content/drive/MyDrive/@antigen_epitope/abdb_combine/data_Y/all_data_with_SI.csv')
df_z = pd.read_csv('/content/drive/MyDrive/@antigen_epitope/abdb_combine/data_Z/all_data_with_SI.csv')

In [ ]:
df_x_clean = df_x[['ResID', 'AA', 'Status']].copy()
df_x_clean['Chain'] = 'X'

df_y_clean = df_y[['ResID', 'AA', 'Status']].copy()
df_y_clean['Chain'] = 'Y'

df_z_clean = df_z[['ResID', 'AA', 'Status']].copy()
df_z_clean['Chain'] = 'Z'

status_df = pd.concat([df_x_clean, df_y_clean, df_z_clean], ignore_index=True)
df = df.merge(status_df, on=['Chain', 'ResID', 'AA'], how='left')

In [ ]:
df['SaSaRate']

In [ ]:
# Define the directory where the merged file will be saved
save_dir = os.path.join(new_fold_path, 'data')
os.makedirs(save_dir, exist_ok=True)

# Save the concatenated dataframe to a CSV file
all_data_with_si_filepath = os.path.join(save_dir, 'all_data_with_SI.csv')
df.to_csv(all_data_with_si_filepath, index=False)
print(f"All data has been merged and saved to {all_data_with_si_filepath}")

In [ ]:
interacting_count = (df['SaSaRate'] > 0.2).sum()
print(interacting_count)

In [ ]:
interacting_count = (df['Status'] == 'Interacting').sum()
print(interacting_count)